# B - Vegetation, tussocks, volume and biomass (B1-B4)

In [ ]:
import sys; sys.path.insert(0, "..")            # so the notebook finds helpers.py
import numpy as np, pandas as pd, matplotlib.pyplot as plt, laspy as las
import helpers as H

EP = H.epochs()                                  # all epochs found in data/merged
R = {e: H.process(e) for e in EP}           # ground / vegetation separation: slow the first time, cached afterwards
print([H.EPOCH_LABEL[e] for e in EP])
E = "20250327" if "20250327" in EP else EP[-1]       # epoch of the manual biomass samples
r = R[E]; t = H.tussocks(r)                          # one row per tussock

## B1 - Vegetation / ground separation applied to the study area

1. Apply your vegetation/ground separation method to the full dataset. Or a portion of it, if too big. Report the fraction of points classified as vegetation, visualize the result, and discuss any obvious misclassification you observe (e.g., low shrubs vs. terrain roughness).
2. Describe and motivate your chosen vegetation characterisation method: what intermediate quan- tity does it produce (e.g., volume, height, area), how is it computed, and what are its key parame- ters/settings? Discuss how sensitive your result is to these choices.
3. Clearly describe how you convert your vegetation structure characterisation into a biomass es- timate: what is it based on (e.g., literature source), and what assumptions does it rely on (e.g., vegetation type, density)? Apply it to estimate biomass for at least three vegetation patches that visually differ in type or size (e.g., from A2), and discuss if the relative differences between them seem logical.
4. Reflect on the main sources of uncertainty in your biomass estimates. Separate clearly between uncertainty through the vegetation structure characterisation (point cloud density / quality, method and parameter choices) and uncertainty in the biomass conversion.


In [ ]:

import laspy
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import binary_dilation

# ---- 1. Load LAS file
project_root = Path.cwd().parent
las_file = project_root / "data" / "20250327-153520-F001-S003_cropped.las"
las = laspy.read(las_file)

x, y, z = np.asarray(las.x), np.asarray(las.y), np.asarray(las.z)

rgb = np.column_stack([las.red, las.green, las.blue]).astype(np.float32)
rgb /= 255 if rgb.max() <= 255 else 65535

# ---- 2. Define bounding box
xmin, xmax = 72590, 72650
ymin, ymax = 451680, 451740

box = (xmin, ymin, xmax, ymax)

# Crop points
mask = (x >= xmin) & (x <= xmax) & (y >= ymin) & (y <= ymax)

x, y, z, rgb = x[mask], y[mask], z[mask], rgb[mask]

print(f"Points in bounding box: {len(x):,}")

if len(x) == 0:
    raise ValueError("No points inside the selected bounding box.")

# ---- 3. Thin points for classification
occupied_m2 = len(np.unique(
    np.floor(x).astype(np.int64) * 10_000_000 +
    np.floor(y).astype(np.int64)
))

keep = np.random.default_rng(0).random(len(x)) < min(
    1.0, H.DENSITY * occupied_m2 / len(x)
)

p = {
    "x": x[keep],
    "y": y[keep],
    "z": z[keep],
    "rgb": rgb[keep]
}

# ---- 4. Vegetation / ground classification
R = H.separate(p, box=box, debug=True)
R["epoch"] = "20250327"

D = R["debug"]

isveg = np.zeros(len(D["x"]), dtype=bool)
isveg[D["veg"]] = True

print(f"Vegetation points: {isveg.sum():,}")
print(f"Ground points: {(~isveg).sum():,}")


# ---- 5. Transfer classification to full-resolution points

from scipy.spatial import cKDTree

# Coordinates of classified points
classified_xyz = np.column_stack([D["x"], D["y"], D["z"]])

# Coordinates of ALL original cropped points
original_xyz = np.column_stack([x, y, z])

# Find nearest classified point for every original point
tree = cKDTree(classified_xyz)
distance, nearest = tree.query(original_xyz, k=1)

# Transfer vegetation labels
full_isveg = isveg[nearest]

print(f"Full-resolution points: {len(full_isveg):,}")
print(f"Vegetation points: {full_isveg.sum():,}")
print(f"Ground points: {(~full_isveg).sum():,}")
print(f"95th percentile matching distance: {np.percentile(distance, 95):.3f} m")

# Save ALL original cropped points
out_las = laspy.LasData(las.header)
out_las.points = las.points[mask]

# Add vegetation classification
out_las.add_extra_dim(
    laspy.ExtraBytesParams(
        name="veg_class",
        type=np.uint8,
        description="0=Ground, 1=Vegetation"
    )
)

out_las.veg_class = full_isveg.astype(np.uint8)

# Export
output_file = project_root / "data" / "20250327_cropped_classified_full.laz"
out_las.write(output_file)

print(f"Saved full-resolution classified LAZ: {output_file}")


# ---- 6. Plot classification and hillshade
fig, ax = plt.subplots(1, 2, figsize=(14, 7))

# (a) Ground vs vegetation
ax[0].scatter(
    D["x"][~isveg], D["y"][~isveg],
    s=0.3, c="lightgrey", rasterized=True
)
ax[0].scatter(
    D["x"][isveg], D["y"][isveg],
    s=0.3, c="green", rasterized=True
)
ax[0].set_title("Ground (grey) vs vegetation (green)")

# (b) Hillshade with vegetation overlay
ax[1].imshow(
    H.hillshade(R),
    extent=H.extent(R["grid_dtm"]),
    cmap="gray"
)

big = binary_dilation(H.cover(R), iterations=4)

ax[1].imshow(
    np.ma.masked_where(~big, big),
    extent=H.extent(R["grid_fine"]),
    cmap="autumn",
    vmin=0,
    vmax=1
)

ax[1].set_title("Hillshade + detected vegetation")

# Format maps
for a in ax:
    a.set_xlim(xmin, xmax)
    a.set_ylim(ymin, ymax)
    a.set_aspect("equal")
    a.set_xlabel("X (m)")
    a.set_ylabel("Y (m)")

plt.tight_layout()
H.save(fig, "classification_hillshade_cropped")
plt.show()


In [ ]:
sl = np.kron(r["slope"], np.ones((5, 5)))[::2, ::2][:r["grid_fine"]["nr"], :r["grid_fine"]["nc"]]      # slope on the 0.10 m grid
ok = H.has_data(r); cls = pd.cut(pd.Series(sl[ok]), [-1, 5, 10, 20, 90], labels=["0-5", "5-10", "10-20", ">20"])
tab = pd.DataFrame({"cover_%": (100 * pd.Series(H.cover(r)[ok]).groupby(cls.values, observed=True).mean()).round(2),
                    "area_m2": (pd.Series(np.ones(ok.sum())).groupby(cls.values, observed=True).sum() * H.FINE_RES ** 2).round(0)})
H.table(tab.reset_index().rename(columns={"index": "slope (deg)"}), "B1_cover_vs_slope")

**To write (B1):** the vegetation fraction, the map, and obvious misclassification: vegetation below ~10 cm is missed; steep slopes and wet or rough sand can give false detections (cover increases with slope in the table: real, or a slope artefact?); poles / people were removed by height and shape rules.

## B2 - Characterisation: tussock volume
Tussocks are found with DBSCAN on the vegetation points (points < 20 cm apart belong together). The intermediate quantity is the **canopy volume**, computed four ways: **column** (canopy height integrated over the footprint, closed at the base - main choice), **convex hull**, **voxels** and **alpha shape**.

In [ ]:
vols = ["vol_column", "vol_hull", "vol_voxel", "vol_alpha"]
tot = t[vols].sum().to_frame("total volume (m3)").T.round(2); tot.insert(0, "tussocks", len(t)); H.table(tot, "B2_volume_totals")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, m in zip(axs, ["vol_hull", "vol_voxel", "vol_alpha"]):
    ax.loglog(t.vol_column, t[m], ".", c="#2a7f3f"); lim = [t.vol_column.min() / 2, t[vols].max().max() * 2]; ax.plot(lim, lim, "k--", lw=.6)
    ax.set_xlabel("column volume (m3)"); ax.set_ylabel(m + " (m3)")
H.save(fig, "B2_estimator_comparison"); t[vols].corr(method="spearman").round(2)

In [ ]:
rows = []                                            # sensitivity to the key parameters
for vox in [0.03, 0.05, 0.10]:
    for ar in [0.10, 0.15, 0.25]:
        s = H.tussocks(r, voxel=vox, alpha_radius=ar); rows.append({"voxel (m)": vox, "alpha radius (m)": ar, "vol_voxel": s.vol_voxel.sum(), "vol_alpha": s.vol_alpha.sum(), "vol_column": s.vol_column.sum()})
H.table(pd.DataFrame(rows).round(2), "B2_sensitivity")

**To write (B2):** column volume is the main estimate (closed base, robust to the parameters); hull over-estimates (fills space between lobes); voxel and alpha volumes change by several times with their size parameter, so they are descriptors, not unbiased volumes (see the sensitivity table).

## B3 - Volume to biomass
**AGB = volume x rho.** rho (kg dry biomass per m3 of canopy volume) is set in `helpers.py` (`RHO`): **placeholder values** until fixed from literature / the manual samples (notebook C).

In [ ]:
show = pd.concat([t.nlargest(1, "vol_column"), t.iloc[[len(t) // 2]], t.nsmallest(1, "vol_column")]).copy(); show["patch"] = ["largest", "medium", "smallest"]
H.table(show[["patch", "footprint_m2", "h_max", "vol_column", "agb_low_kg", "agb_mid_kg", "agb_high_kg"]].round(3), "B3_patches")

**To write (B3):** where rho comes from and its assumptions (species, dry vs fresh mass, canopy density). Are the differences between the patches logical (bigger plant = more biomass; similar plants = similar kg per m2)?

## B4 - Uncertainty
Two blocks: **structure** (how tussocks are found / measured: DBSCAN distance, minimum points, minimum height) and **conversion** (rho). Every combination is run.

In [ ]:
import itertools
rows = []
for eps, ms, mh in itertools.product([0.15, 0.20, 0.30], [5, 8, 12], [0.08, 0.10, 0.15]):
    s = H.tussocks(r, eps=eps, min_samples=ms, min_height=mh, alpha=False); rows.append({"eps": eps, "min_samples": ms, "min_height": mh, "tussocks": len(s), "volume_m3": s.vol_column.sum()})
sw = H.table(pd.DataFrame(rows).round(3), "B4_parameter_sweep")
base = t.vol_column.sum()
unc = pd.DataFrame({"low": [sw.volume_m3.min(), base * H.RHO["low"]], "base": [base, base * H.RHO["mid"]], "high": [sw.volume_m3.max(), base * H.RHO["high"]]},
                   index=["structure: volume (m3) over all parameter sets", "conversion: biomass (kg) for rho low / mid / high"]).round(2)
H.table(unc.reset_index(), "B4_uncertainty_budget")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, p in zip(axs, ["eps", "min_samples", "min_height"]):
    sw.groupby(p).volume_m3.agg(["min", "median", "max"]).plot(ax=ax, marker="o"); ax.set_title("effect of " + p)
axs[0].set_ylabel("total volume (m3)"); H.save(fig, "B4_parameter_effects")

**To write (B4):** which block dominates (compare the relative spread of the two rows) and what that means. Also name density / noise (structure) and species, moisture and dry vs fresh mass (conversion).